# 🤖 Notebook 03: Klasifikasi Data
## Analitika Data Keuangan Sektor Publik

**Tujuan Pembelajaran:**
1. Menyiapkan data untuk klasifikasi
2. Melatih model Decision Tree, KNN, dan Naive Bayes
3. Mengevaluasi dan membandingkan performa model
4. Memvisualisasikan confusion matrix dan decision tree
5. Menginterpretasikan hasil model dalam konteks keuangan daerah

---
> **Problem:** Prediksi **Predikat Kinerja** pemerintah daerah (Kurang/Cukup/Baik/Sangat Baik) berdasarkan profil keuangan APBD-nya.

## 📦 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')

print('Library berhasil dimuat ✅')

## 📂 2. Persiapan Data

In [ ]:
# Muat dan bersihkan data
df = pd.read_csv('../../Dataset/02-EDA/keuangan_pemda.csv')

# Hapus nilai tidak wajar
df = df[df['Anggaran_APBD'] > 0].copy()
df.dropna(subset=['Anggaran_APBD', 'Realisasi_APBD', 'Predikat'], inplace=True)

# Feature Engineering
df['Pct_Realisasi'] = df['Realisasi_APBD'] / df['Anggaran_APBD'] * 100
df['Rasio_Kemandirian'] = df['PAD'] / df['Anggaran_APBD'] * 100
df['Rasio_Belanja_Pegawai'] = df['Belanja_Pegawai'] / df['Realisasi_APBD'] * 100
df['Rasio_Belanja_Modal'] = df['Belanja_Modal'] / df['Realisasi_APBD'] * 100

# Definisi Feature dan Target
feature_cols = ['Pct_Realisasi', 'Rasio_Kemandirian', 'Rasio_Belanja_Pegawai',
                'Rasio_Belanja_Modal', 'PAD', 'Anggaran_APBD', 'Realisasi_APBD']
target_col = 'Predikat'

X = df[feature_cols].fillna(0)
y = df[target_col]

print(f'Dataset siap: {X.shape[0]} sampel, {X.shape[1]} fitur')
print(f'\nDistribusi kelas:')
print(y.value_counts())

In [ ]:
# Normalisasi & Split Data Train/Test
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Data Training : {X_train.shape[0]} sampel ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Data Testing  : {X_test.shape[0]} sampel ({X_test.shape[0]/len(X)*100:.1f}%)')

## 🌳 3. Model 1: Decision Tree

In [ ]:
# Latih Decision Tree
dt = DecisionTreeClassifier(max_depth=4, random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print('DECISION TREE — Hasil Evaluasi')
print('=' * 45)
print(f'Accuracy: {accuracy_score(y_test, y_pred_dt):.4f} ({accuracy_score(y_test, y_pred_dt)*100:.1f}%)')
print()
print(classification_report(y_test, y_pred_dt))

In [ ]:
# Visualisasi Decision Tree
plt.figure(figsize=(18, 8))
class_names = [str(c) for c in dt.classes_]
plot_tree(dt, feature_names=feature_cols, class_names=class_names,
          filled=True, rounded=True, fontsize=9, impurity=False)
plt.title('Decision Tree — Prediksi Predikat Kinerja PEMDA\n(max_depth=4)', 
          fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance — Feature mana paling berpengaruh?
importances = pd.Series(dt.feature_importances_, index=feature_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#e74c3c' if v == importances.max() else '#3498db' for v in importances.values]
importances.plot(kind='barh', ax=ax, color=colors)
ax.set_title('Feature Importance — Decision Tree', fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()
print('Feature paling penting:', importances.idxmax())

## 👥 4. Model 2: K-Nearest Neighbor (KNN)

In [ ]:
# Cari nilai K optimal
k_values = range(1, 21)
train_scores, test_scores = [], []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, knn.predict(X_train)))
    test_scores.append(accuracy_score(y_test, knn.predict(X_test)))

best_k = k_values[np.argmax(test_scores)]

plt.figure(figsize=(10, 5))
plt.plot(k_values, train_scores, 'b-o', label='Training Accuracy', markersize=6)
plt.plot(k_values, test_scores, 'r-o', label='Testing Accuracy', markersize=6)
plt.axvline(best_k, color='green', linestyle='--', label=f'Best K = {best_k}')
plt.xlabel('Nilai K')
plt.ylabel('Accuracy')
plt.title('KNN: Mencari Nilai K Optimal', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()
print(f'K terbaik: {best_k} (Test Accuracy: {max(test_scores):.4f})')

In [ ]:
# Latih KNN dengan K terbaik
knn = KNeighborsClassifier(n_neighbors=best_k)
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)

print(f'KNN (K={best_k}) — Hasil Evaluasi')
print('=' * 45)
print(f'Accuracy: {accuracy_score(y_test, y_pred_knn):.4f} ({accuracy_score(y_test, y_pred_knn)*100:.1f}%)')
print()
print(classification_report(y_test, y_pred_knn))

## 📊 5. Model 3: Naive Bayes

In [ ]:
# Latih Naive Bayes
nb = GaussianNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

print('NAIVE BAYES — Hasil Evaluasi')
print('=' * 45)
print(f'Accuracy: {accuracy_score(y_test, y_pred_nb):.4f} ({accuracy_score(y_test, y_pred_nb)*100:.1f}%)')
print()
print(classification_report(y_test, y_pred_nb))

## ⚖️ 6. Perbandingan & Evaluasi Model

In [ ]:
# Confusion Matrix — semua model
models = {
    'Decision Tree': y_pred_dt,
    f'KNN (K={best_k})': y_pred_knn,
    'Naive Bayes': y_pred_nb
}

class_labels = [c for c in ['Kurang', 'Cukup', 'Baik', 'Sangat Baik'] if c in y.unique()]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, y_pred) in zip(axes, models.items()):
    cm = confusion_matrix(y_test, y_pred, labels=class_labels)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=class_labels, yticklabels=class_labels)
    acc = accuracy_score(y_test, y_pred)
    ax.set_title(f'{name}\nAccuracy: {acc:.2%}', fontweight='bold')
    ax.set_xlabel('Prediksi')
    ax.set_ylabel('Aktual')

plt.suptitle('Confusion Matrix — Perbandingan Algoritma', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Tabel perbandingan performa model
results = []
for name, y_pred in models.items():
    results.append({
        'Model': name,
        'Accuracy': f'{accuracy_score(y_test, y_pred):.4f}',
        'Macro Precision': f'{pd.DataFrame(classification_report(y_test, y_pred, output_dict=True)).T["precision"]["macro avg"]:.4f}',
        'Macro Recall': f'{pd.DataFrame(classification_report(y_test, y_pred, output_dict=True)).T["recall"]["macro avg"]:.4f}',
        'Macro F1': f'{pd.DataFrame(classification_report(y_test, y_pred, output_dict=True)).T["f1-score"]["macro avg"]:.4f}'
    })

hasil_df = pd.DataFrame(results)
print('PERBANDINGAN PERFORMA MODEL')
print('=' * 70)
print(hasil_df.to_string(index=False))
print()
print('✅ Model terbaik berdasarkan Accuracy:', hasil_df.loc[hasil_df['Accuracy'].idxmax(), 'Model'])

## 🔮 7. Prediksi Data Baru

In [ ]:
# Prediksi untuk PEMDA baru
# Feature: [Pct_Realisasi, Rasio_Kemandirian, Rasio_Belanja_Pegawai,
#            Rasio_Belanja_Modal, PAD, Anggaran_APBD, Realisasi_APBD]

# Contoh data PEMDA baru
pemda_baru = pd.DataFrame({
    'Pct_Realisasi': [92.5, 55.0, 75.3],
    'Rasio_Kemandirian': [35.0, 8.0, 18.5],
    'Rasio_Belanja_Pegawai': [35.0, 60.0, 45.0],
    'Rasio_Belanja_Modal': [25.0, 10.0, 20.0],
    'PAD': [3500000000, 500000000, 1800000000],
    'Anggaran_APBD': [10000000000, 6000000000, 9700000000],
    'Realisasi_APBD': [9250000000, 3300000000, 7300000000]
})

pemda_baru_scaled = scaler.transform(pemda_baru)

print('PREDIKSI PREDIKAT KINERJA PEMDA BARU')
print('=' * 55)
print(f'{'PEMDA':8} | {'Decision Tree':14} | {'KNN':8} | {'Naive Bayes':12} | {'Aktual (hitung)':15}')
print('-' * 65)

for i, (idx, row) in enumerate(pemda_baru.iterrows()):
    # Hitung aktual berdasarkan aturan bisnis
    pct = row['Pct_Realisasi']
    if pct >= 90: aktual = 'Sangat Baik'
    elif pct >= 80: aktual = 'Baik'
    elif pct >= 60: aktual = 'Cukup'
    else: aktual = 'Kurang'

    pred_dt = dt.predict([pemda_baru_scaled[i]])[0]
    pred_knn = knn.predict([pemda_baru_scaled[i]])[0]
    pred_nb = nb.predict([pemda_baru_scaled[i]])[0]
    print(f'PEMDA{i+1:3}  | {pred_dt:14} | {pred_knn:8} | {pred_nb:12} | {aktual}')

## 📝 8. Latihan

1. Ubah `max_depth` Decision Tree menjadi 3, 5, 6. Apa dampaknya terhadap accuracy?
2. Tambahkan fitur `Tahun` dalam feature set. Apakah meningkatkan performa?
3. Coba gunakan **stratified 5-fold cross validation** untuk evaluasi yang lebih robust.
4. Buat prediksi untuk data pemda Anda sendiri (buat angka realistis).
5. Algoritma mana yang paling baik untuk kasus ini? Jelaskan alasannya.

In [ ]:
# ✏️ Cross Validation (starter)
for name, model in [('Decision Tree', dt), (f'KNN K={best_k}', knn), ('Naive Bayes', nb)]:
    scores = cross_val_score(model, X_scaled, y, cv=5, scoring='accuracy')
    print(f'{name:20}: Mean={scores.mean():.4f}, Std={scores.std():.4f}, Scores={scores.round(3)}')